# RQ5: 非典型性与学术生涯留存

研究问题：数据使用非典型性如何影响学者的学术生涯留存？

因变量：Academic Retention（生存/离职）
核心自变量：Atypicality
控制变量：H_index, Publication_Count, Academic_Age, Topic_Diversity

模型：Cox 比例风险模型 + Kaplan-Meier 生存曲线

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

from src.data_loader import load_config
from src.regression import run_cox_ph
from src.visualization import plot_km_curve, save_fig
import pandas as pd

In [ ]:
config = load_config()
df = pd.read_csv(os.path.join('studies', 'RQ5', 'rq5_regression_ready.csv'))
print(f'数据维度: {df.shape}')
df.head()

## 1. Cox 比例风险模型

In [ ]:
x_var = 'Atypicality'
controls = ['H_index', 'Publication_Count', 'Academic_Age', 'Topic_Diversity']
controls = [c for c in controls if c in df.columns]

cox_result = run_cox_ph(df, duration_var='Academic_Duration', event_var='Retained',
                        x_var=x_var, controls=controls)
if cox_result['converged']:
    print(f'HR = {np.exp(cox_result["coef"]):.4f}')
    print(f'p-value = {cox_result["pvalue"]:.4f}')
    print(f'95% CI: [{np.exp(cox_result["ci_lower"]):.4f}, {np.exp(cox_result["ci_upper"]):.4f}]')

## 2. Kaplan-Meier 生存曲线

In [ ]:
df['Atypicality_Group'] = pd.qcut(df['Atypicality'], q=3, labels=['Low', 'Medium', 'High'])
fig = plot_km_curve(df, duration_col='Academic_Duration', event_col='Retained',
                    group_col='Atypicality_Group',
                    title='RQ5: Survival by Atypicality Level')
save_fig(fig, os.path.join('studies', 'RQ5', 'figures', 'rq5_km_curve.pdf'))
plt.show()